In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)


e:\health_ai_system_with_code\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TensorFlow version: 2.20.0
NumPy version: 2.2.6


In [3]:
import sys
import os

# Add project root to Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

print("Project root added to sys.path:", PROJECT_ROOT)


Project root added to sys.path: e:\health_ai_system_with_code


In [5]:
from utils.config import IMAGE_SIZE, BATCH_SIZE, EPOCHS

print("IMAGE_SIZE:", IMAGE_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)


IMAGE_SIZE: (224, 224)
BATCH_SIZE: 32
EPOCHS: 10


In [7]:
!pip install kagglehub




   ---------------------------------------- 0/2 [pyyaml]
   -------------------- ------------------- 1/2 [kagglehub]
   -------------------- ------------------- 1/2 [kagglehub]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]



In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kritikseth/fruit-and-vegetable-image-recognition")

print("Path to dataset files:", path)

e:\health_ai_system_with_code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.98G/1.98G [07:44<00:00, 4.59MB/s]


Extracting files...
Path to dataset files: C:\Users\sudeep1\.cache\kagglehub\datasets\kritikseth\fruit-and-vegetable-image-recognition\versions\8


In [9]:
import shutil
import os

# path variable already created by kagglehub
source_dir = os.path.join(path, "train")
target_dir = "data/raw/food_images"

os.makedirs(target_dir, exist_ok=True)

for folder in os.listdir(source_dir):
    src = os.path.join(source_dir, folder)
    dst = os.path.join(target_dir, folder)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

print("Dataset copied to project folder successfully!")


Dataset copied to project folder successfully!


In [10]:
import os

print("Classes found:", os.listdir("data/raw/food_images"))


Classes found: ['apple', 'banana', 'beetroot', 'bell pepper', 'cabbage', 'capsicum', 'carrot', 'cauliflower', 'chilli pepper', 'corn', 'cucumber', 'eggplant', 'garlic', 'ginger', 'grapes', 'jalepeno', 'kiwi', 'lemon', 'lettuce', 'mango', 'onion', 'orange', 'paprika', 'pear', 'peas', 'pineapple', 'pomegranate', 'potato', 'raddish', 'soy beans', 'spinach', 'sweetcorn', 'sweetpotato', 'tomato', 'turnip', 'watermelon']


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    "data/raw/food_images",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_generator = train_datagen.flow_from_directory(
    "data/raw/food_images",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

print("Class indices:", train_generator.class_indices)
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)


Found 2510 images belonging to 36 classes.
Found 605 images belonging to 36 classes.
Class indices: {'apple': 0, 'banana': 1, 'beetroot': 2, 'bell pepper': 3, 'cabbage': 4, 'capsicum': 5, 'carrot': 6, 'cauliflower': 7, 'chilli pepper': 8, 'corn': 9, 'cucumber': 10, 'eggplant': 11, 'garlic': 12, 'ginger': 13, 'grapes': 14, 'jalepeno': 15, 'kiwi': 16, 'lemon': 17, 'lettuce': 18, 'mango': 19, 'onion': 20, 'orange': 21, 'paprika': 22, 'pear': 23, 'peas': 24, 'pineapple': 25, 'pomegranate': 26, 'potato': 27, 'raddish': 28, 'soy beans': 29, 'spinach': 30, 'sweetcorn': 31, 'sweetpotato': 32, 'tomato': 33, 'turnip': 34, 'watermelon': 35}
Training samples: 2510
Validation samples: 605


In [12]:
import tensorflow as tf

# Load pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze base model
base_model.trainable = False

# Build full model
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(train_generator.num_classes, activation="softmax")
])

# Compile model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,596 (9.26 MB)

 Trainable params: 168,612 (658.64 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)


e:\health_ai_system_with_code\.venv\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - accuracy: 0.4968 - loss: 1.9300 - val_accuracy: 0.7438 - val_loss: 0.7492
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.7709 - loss: 0.7712 - val_accuracy: 0.7950 - val_loss: 0.6071
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.8255 - loss: 0.5604 - val_accuracy: 0.7868 - val_loss: 0.5729
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.8717 - loss: 0.4006 - val_accuracy: 0.8198 - val_loss: 0.5023
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.8940 - loss: 0.3285 - val_accuracy: 0.8182 - val_loss: 0.4922
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.9159 - loss: 0.2634 - val_accuracy: 0.8198 - val_loss: 0.5047
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.9382 - loss: 0.2134 - val_accuracy: 0.8298 - val_loss: 0.5161
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.9418 - loss: 0.1815 - val_accuracy: 0.8215 - val_loss

In [14]:
model.save("models/food_model/food_classifier.h5")
print("Model saved successfully!")


Model saved successfully!


In [15]:
import json

class_indices = train_generator.class_indices

with open("models/food_model/class_indices.json", "w") as f:
    json.dump(class_indices, f)

print("Class indices saved:", class_indices)


Class indices saved: {'apple': 0, 'banana': 1, 'beetroot': 2, 'bell pepper': 3, 'cabbage': 4, 'capsicum': 5, 'carrot': 6, 'cauliflower': 7, 'chilli pepper': 8, 'corn': 9, 'cucumber': 10, 'eggplant': 11, 'garlic': 12, 'ginger': 13, 'grapes': 14, 'jalepeno': 15, 'kiwi': 16, 'lemon': 17, 'lettuce': 18, 'mango': 19, 'onion': 20, 'orange': 21, 'paprika': 22, 'pear': 23, 'peas': 24, 'pineapple': 25, 'pomegranate': 26, 'potato': 27, 'raddish': 28, 'soy beans': 29, 'spinach': 30, 'sweetcorn': 31, 'sweetpotato': 32, 'tomato': 33, 'turnip': 34, 'watermelon': 35}


In [1]:
import tensorflow as tf
import json

# Load trained model
model = tf.keras.models.load_model(
    "models/food_model/food_classifier.h5"
)

# Load class labels
with open("models/food_model/class_indices.json", "r") as f:
    class_indices = json.load(f)

# Reverse mapping (index → class name)
index_to_class = {v: k for k, v in class_indices.items()}

print("Model and labels loaded successfully!")
print(index_to_class)


e:\health_ai_system_with_code\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Model and labels loaded successfully!
{0: 'apple', 1: 'banana', 2: 'beetroot', 3: 'bell pepper', 4: 'cabbage', 5: 'capsicum', 6: 'carrot', 7: 'cauliflower', 8: 'chilli pepper', 9: 'corn', 10: 'cucumber', 11: 'eggplant', 12: 'garlic', 13: 'ginger', 14: 'grapes', 15: 'jalepeno', 16: 'kiwi', 17: 'lemon', 18: 'lettuce', 19: 'mango', 20: 'onion', 21: 'orange', 22: 'paprika', 23: 'pear', 24: 'peas', 25: 'pineapple', 26: 'pomegranate', 27: 'potato', 28: 'raddish', 29: 'soy beans', 30: 'spinach', 31: 'sweetcorn', 32: 'sweetpotato', 33: 'tomato', 34: 'turnip', 35: 'watermelon'}


In [2]:
import cv2
import numpy as np

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    return img


In [3]:
def predict_food(image_path):
    img = preprocess_image(image_path)
    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    confidence = float(np.max(prediction))
    food_name = index_to_class[class_index]
    return food_name, confidence


In [5]:
import os

print(os.path.exists("data/raw/food_images/apple"))
print(os.listdir("data/raw/food_images/apple"))


True
['Image_1.jpg', 'Image_10.jpg', 'Image_16.jpg', 'Image_17.jpg', 'Image_18.jpg', 'Image_19.jpg', 'Image_2.jpg', 'Image_20.jpg', 'Image_21.jpg', 'Image_23.jpg', 'Image_24.jpg', 'Image_25.jpg', 'Image_26.jpg', 'Image_27.jpg', 'Image_28.jpg', 'Image_3.jpg', 'Image_31.jpg', 'Image_32.jpg', 'Image_33.jpg', 'Image_34.jpg', 'Image_35.png', 'Image_36.jpg', 'Image_37.jpg', 'Image_38.jpg', 'Image_39.jpg', 'Image_40.jpg', 'Image_41.jpg', 'Image_42.jpg', 'Image_43.jpg', 'Image_44.jpg', 'Image_45.jpg', 'Image_47.jpg', 'Image_48.jpg', 'Image_49.jpg', 'Image_5.JPG', 'Image_50.jpg', 'Image_51.jpg', 'Image_52.jpg', 'Image_53.png', 'Image_54.jpg', 'Image_55.jpg', 'Image_56.jpg', 'Image_57.jpg', 'Image_58.jpg', 'Image_59.png', 'Image_6.jpg', 'Image_60.jpg', 'Image_61.jpg', 'Image_62.jpg', 'Image_63.jpg', 'Image_64.jpg', 'Image_65.png', 'Image_67.jpg', 'Image_68.jpg', 'Image_69.jpg', 'Image_7.jpg', 'Image_71.png', 'Image_76.png', 'Image_78.jpg', 'Image_80.jpg', 'Image_81.png', 'Image_82.jpg', 'Image_8

In [6]:
test_image = "data/raw/food_images/apple/Image_1.jpg"

food, confidence = predict_food(test_image)
print(f"Predicted food: {food}")
print(f"Confidence: {confidence:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 896ms/step
Predicted food: apple
Confidence: 0.83
